In [1]:
import torch
import torchaudio

print(torch.__version__)
print(torchaudio.__version__)

2.8.0+cu128
2.8.0+cu128


In [2]:
from fairseq2 import gang
gang._thread_local.current_gangs = []

In [3]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
pipeline = ASRInferencePipeline(model_card= 'omniASR_LLM_300M')

Output()

In [4]:
target_langs = ['eng_Latn', 'swh_Latn']

In [61]:
import subprocess
from pathlib import Path

def encode_to_wav(audio):
    encoded_audio = subprocess.run(
        ['ffmpeg', '-i', audio, '-f', 'wav', 'pipe:1'],
        check = True,
        capture_output= True
    )
    return encoded_audio.stdout #stdout is the actual file

audio = Path('../data/Courtroom With Sarah Sothenes.m4a')
encoded_audio = encode_to_wav(audio)
type(encoded_audio) #type(encoded_audio)

bytes

In [62]:
import io, soundfile as sf
#obtained bytes go to a memory like object

audio = io.BytesIO(encoded_audio)
audio.seek(0)

waveform, sr = sf.read(audio)
waveform = torch.from_numpy(waveform).float()

In [63]:
print(f'Sample rate: {sr} Hz')

Sample rate: 48000 Hz


In [8]:
import tempfile

with tempfile.NamedTemporaryFile(suffix='.wav', delete= False) as tmp:
    sf.write(tmp.name, waveform.numpy(), sr)
    transcript = pipeline.transcribe([tmp.name], batch_size= 1)

print(transcript)

['mimi anaitwa nathan una akiukweli napenda walisamaki yaani akiongelea walisamaki naongelea ni rosten ni ule wale ambao yaani samaki wake unakuwa unaoroja uroja yaani unakuwa mtaa tunaopenda']


In [64]:
def segment_audio(waveform, sr, chunk_duration= 5, overlap= 0.5):
    """
    Args:
        waveform: audio data in torch.Tensor format
        sr: sample rate i.e, number of audio samples in a second
        chunk_duration: how long a chunk is, defaults to 5 as defined in this function
        overlap: overlap ratio (0-1), eg. o.5 overlap means 50% overlap

    Returns:
        List of tuples, containing the chunk data, it's start time and end time
    """
    chunk_size = int(sr * chunk_duration)
    hop_size = int(chunk_size * (1 - overlap)) #stride between chunks

    chunks = []
    start_idx = 0

    while start_idx < len(waveform):
        end_idx = min(start_idx + chunk_size, len(waveform)) #for the last chunk, it's usually not the full chunk size
        chunk = waveform[start_idx:end_idx]

        start_time = start_idx / sr
        end_time = end_idx / sr

        chunks.append((chunk, start_time, end_time))
        start_idx += hop_size

    return chunks

In [65]:
chunks = segment_audio(waveform, sr)
chunks[-1]

(tensor([[-0.0766, -0.0766],
         [-0.0771, -0.0771],
         [-0.0766, -0.0766],
         ...,
         [-0.0769, -0.0769],
         [-0.0699, -0.0699],
         [-0.0622, -0.0622]]),
 2997.5,
 2998.826666666667)

In [66]:
transcripts = []
for chunk, start_time, end_time in chunks:
    with tempfile.NamedTemporaryFile(suffix= '.wav', delete= False) as tmp:
        sf.write(tmp.name, chunk.numpy(), sr)
        transcript = pipeline.transcribe([tmp.name], batch_size= 1, lang= ['swh_Latn'])[0]
        transcripts.append({
            'text': transcript,
            'start': start_time,
            'end': end_time,
        })
        print(f"{start_time:.1f}s - {end_time:.1f}s: {transcript}")

0.0s - 5.0s: wakavinena kuwa wenye hekima walipumbazika
2.5s - 7.5s: walipumba ziko wakaugabiwa utukufua mungu asi-
5.0s - 10.0s: wakaubadili utukufu wa mungu asile wa vile kuconfirm ndiyo inasura vile damu
7.5s - 12.5s: na wari wiki kwa mfano na sura vile damu aliye na wari wiki na na jengeni na
10.0s - 15.0s: aliye na waligifu na na ndege na wanyama na vitu vitabu
12.5s - 17.5s: manyama na vitu vitamangu
15.0s - 20.0s: percentage six
17.5s - 22.5s: thirty six hiyo hiyo mungu aliwaachwa kwanzi ta-
20.0s - 25.0s: hivyo mungu aliwaacha wafuate ta- tambaa za- zao zaidi
22.5s - 27.5s: namba za- zao zaidi hata wanawake wakabarili ma-
25.0s - 30.0s: hata wanawake wakabarili ma- matumizi asili kwa matumizi asili
27.5s - 32.5s: watu mwizi asili kwa watu mwizi asi- asili wanaenda
30.0s - 35.0s: nzio ya simu kwenda umelipwa hivyo waliacha na tu
32.5s - 37.5s: ume iko hivyo waliacha na kiuzi aniche kiasili wakaa
35.0s - 40.0s: ziano kiasi waka- wakawaki- kiana tambaa wanaume
37.5s - 42.5s: waki-

KeyboardInterrupt: 

In [58]:
[item['text'] for item in transcripts]

['mimi anaitwa nasa na kiukweli napenda wali sanaki',
 'ukweli napenda walisamaki yaani nikiongelea walisamaki',
 'ni kiongelea walisamaki naongelea ni rosteni',
 'naongelea ni rosten ni ule wale ambao yanisema kila',
 'ni ule wale ambao yani samaki wake unakuwa uneno unaroja uroja yaani unakuwa',
 'lakini unakuwa unaenda unaroja uroja yaani unakuwa mtaa tunaopenda',
 'tamu tunaopenda',
 'unini']

In [59]:
def merge_transcripts(transcripts):
    if not transcripts:
        return ""
    merged = transcripts[0]['text']

    for i in range(1, len(transcripts)):
        prev_end = transcripts[i-1]['end']
        curr_start = transcripts[i]['start']
        curr_text = transcripts[i]['text']

        #Add non overlapping part if there's overlap
        if curr_start < prev_end:
            #The overlap divided by total length of segment
            overlap_ratio = (prev_end - curr_start) / (transcripts[i]['end'] - curr_start)
            words = curr_text.split()
            skip_words = int(len(words) * overlap_ratio)
            non_overlapping = " ".join(words[skip_words:])
            merged += " "+ non_overlapping

        else:
            merged += " "+ curr_text
    return merged

In [60]:
all_transcripts= merge_transcripts(transcripts)
all_transcripts

'mimi anaitwa nasa na kiukweli napenda wali sanaki yaani nikiongelea walisamaki naongelea ni rosteni ule wale ambao yanisema kila wake unakuwa uneno unaroja uroja yaani unakuwa uroja yaani unakuwa mtaa tunaopenda tunaopenda '